# 2. Event study around major and sponsored streams

Define events (a stream above a viewer threshold, or one with `sponsored_flag`), then track player counts and new reviews in the hours before and after. Check pre-trends and control for Google Trends interest (hand-downloaded CSVs in `data/raw/google_trends/`).

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
COLLECTED = ROOT / "data" / "collected"
con = duckdb.connect()

In [ ]:
steam = con.sql(f"SELECT * FROM read_csv_auto('{COLLECTED}/**/steam_*.csv', union_by_name=true)").df()
steam["snapshot_utc"] = pd.to_datetime(steam["snapshot_utc"])

In [ ]:
twitch = con.sql(f"SELECT * FROM read_csv_auto('{COLLECTED}/**/twitch_*.csv', union_by_name=true)").df()
twitch["snapshot_utc"] = pd.to_datetime(twitch["snapshot_utc"])

In [ ]:
# Candidate events: first snapshot of each stream that crosses the size threshold or is flagged sponsored
VIEWER_THRESHOLD = 20_000
events = (twitch[(twitch.viewer_count >= VIEWER_THRESHOLD) | twitch.sponsored_flag]
          .sort_values('snapshot_utc').groupby('stream_id').first().reset_index())
events[['snapshot_utc', 'twitch_game_name', 'user_login', 'viewer_count', 'sponsored_flag']].head(20)

In [ ]:
# TODO: event-time panel (hours -24..+72), diff-diff event study with game and hour-of-week fixed effects,
# not-yet-treated games as controls, and a pre-trend test.